# Knowledge Distillation Experiment: Image Classification (MNIST)

## Experiment Goals:
1. Train a large CNN teacher on MNIST
2. Train a small CNN student using distillation
3. Compare with a baseline student (no distillation)
4. Analyze performance vs model size tradeoff

## Setup and Dependencies

In [ ]:
# setup: using keras/tensorflow instead of pytorch (pytorch has compatibility issues)
# uncomment below if tensorflow not installed
#!pip install tensorflow

import numpy as np
import matplotlib.pyplot as plt
from tensorflow import keras
import tensorflow as tf
import time

# set random seeds for reproducibility
np.random.seed(42)
tf.random.set_seed(42)

print(f"tensorflow version: {tf.__version__}")
print(f"keras version: {keras.__version__}")

## Load MNIST Dataset

In [ ]:
# mnist: 60k train and 10k test 28x28 grayscale digits, 10 classes
print("downloading MNIST with keras...")

(x_train, y_train), (x_test, y_test) = keras.datasets.mnist.load_data()

x_train = x_train.astype('float32') / 255.0
x_test = x_test.astype('float32') / 255.0

# add channel dimension (keras uses channels_last format)
x_train = np.expand_dims(x_train, -1)
x_test = np.expand_dims(x_test, -1)

In [ ]:
print(f"training samples: {len(x_train)}")
print(f"test samples: {len(x_test)}")
print(f"training data shape: {x_train.shape}")
print(f"test data shape: {x_test.shape}")

a few training digits with their labels

In [ ]:
fig, axes = plt.subplots(2, 5, figsize=(12, 5))
for i, ax in enumerate(axes.flat):
    img = x_train[i].squeeze()
    label = y_train[i]
    
    ax.imshow(img, cmap='gray')
    ax.set_title(f'label: {label}')
    ax.axis('off')
plt.tight_layout()
plt.show()

## Define Teacher and Student Models

In [ ]:
# teacher: a large cnn, only its accuracy matters because it is never deployed
def create_teacher_model():
    model = keras.Sequential([
        keras.layers.Input(shape=(28, 28, 1)),
        keras.layers.Conv2D(64, 3, padding='same', activation='relu'),
        keras.layers.MaxPooling2D(2),
        keras.layers.Conv2D(128, 3, padding='same', activation='relu'),
        keras.layers.MaxPooling2D(2),
        keras.layers.Conv2D(256, 3, padding='same', activation='relu'),
        keras.layers.MaxPooling2D(2),
        keras.layers.Flatten(),
        keras.layers.Dense(512, activation='relu'),
        keras.layers.Dropout(0.5),
        keras.layers.Dense(256, activation='relu'),
        keras.layers.Dropout(0.5),
        # no softmax: the model emits logits so the distillation loss can soften them with softmax(z / t)
        keras.layers.Dense(10),
    ])
    return model

In [ ]:
teacher = create_teacher_model()
teacher.summary()

In [ ]:
# student: about 8x fewer parameters than the teacher, the model meant for deployment
def create_student_model():
    model = keras.Sequential([
        keras.layers.Input(shape=(28, 28, 1)),
        keras.layers.Conv2D(16, 3, padding='same', activation='relu'),
        keras.layers.MaxPooling2D(2),
        keras.layers.Conv2D(32, 3, padding='same', activation='relu'),
        keras.layers.MaxPooling2D(2),
        keras.layers.Flatten(),
        keras.layers.Dense(128, activation='relu'),
        keras.layers.Dropout(0.3),
        # no softmax: the model emits logits so the distillation loss can soften them with softmax(z / t)
        keras.layers.Dense(10),
    ])
    return model

In [ ]:
student = create_student_model()
student.summary()

In [ ]:
teacher_params = teacher.count_params()
student_params = student.count_params()

print("model architecture comparison:")
print(f"teacher parameters: {teacher_params:,}")
print(f"student parameters: {student_params:,}")
print(f"compression ratio: {teacher_params/student_params:.1f}x")
print(f"size reduction: {(1-student_params/teacher_params)*100:.1f}%")

## Training Functions

In [ ]:
# distillation loss after hinton, vinyals and dean (2015): alpha weights the soft term, 1 - alpha the hard term
def distillation_loss(y_true, student_logits, teacher_logits, temperature, alpha):
    # hard term at t=1: cross entropy with the true labels keeps the student anchored to the ground truth
    hard = tf.reduce_mean(tf.nn.sparse_softmax_cross_entropy_with_logits(
        labels=tf.cast(y_true, tf.int32), logits=student_logits))
    # soft term: kl(teacher_t || student_t) on logits, a temperature above 1 exposes how the teacher ranks
    # the wrong classes, log space avoids an epsilon, times t^2 because soft target gradients shrink as 1 / t^2
    teacher_log = tf.nn.log_softmax(teacher_logits / temperature)
    student_log = tf.nn.log_softmax(student_logits / temperature)
    soft = tf.reduce_mean(tf.reduce_sum(tf.exp(teacher_log) * (teacher_log - student_log), axis=1)) * temperature ** 2
    return alpha * soft + (1 - alpha) * hard

## Train Teacher Model

In [ ]:
# shared by the teacher, both students and the timing benchmark
batch_size = 128
print(f"batch size: {batch_size}")

In [ ]:
# the teacher is trained first, on hard labels only, then only read during distillation
print("training teacher model...")

In [ ]:
teacher = create_teacher_model()
teacher.compile(
    optimizer=keras.optimizers.Adam(learning_rate=0.001),
    loss=keras.losses.SparseCategoricalCrossentropy(from_logits=True),
    metrics=['accuracy'])

In [ ]:
history_teacher = teacher.fit(
    x_train, y_train,
    validation_data=(x_test, y_test),
    epochs=10,
    batch_size=batch_size,
    verbose=1)

In [ ]:
teacher_losses = history_teacher.history['loss']
teacher_accs = [acc * 100 for acc in history_teacher.history['val_accuracy']]

print(f"\nfinal teacher accuracy: {teacher_accs[-1]:.2f}%")

## train student with distillation

In [ ]:
print("training student with knowledge distillation")

# custom loop because fit() has no hook for feeding teacher outputs into the loss
student_distilled = create_student_model()
optimizer = keras.optimizers.Adam(learning_rate=0.001)

temperature = 4.0
alpha = 0.7

distill_losses = []
distill_accs = []

epochs = 10
steps_per_epoch = len(x_train) // batch_size

In [ ]:
for epoch in range(epochs):
    print(f"\nepoch {epoch+1}/{epochs}")
    epoch_loss = 0
    epoch_acc = 0
    
    indices = np.random.permutation(len(x_train))
    x_train_shuffled = x_train[indices]
    y_train_shuffled = y_train[indices]
    
    for step in range(steps_per_epoch):
        start_idx = step * batch_size
        end_idx = start_idx + batch_size
        
        x_batch = x_train_shuffled[start_idx:end_idx]
        y_batch = y_train_shuffled[start_idx:end_idx]
        
        with tf.GradientTape() as tape:
            # inference mode keeps the teacher's dropout off, it is frozen because only student variables get gradients
            teacher_logits = teacher(x_batch, training=False)
            
            student_logits = student_distilled(x_batch, training=True)
            
            loss = distillation_loss(y_batch, student_logits, teacher_logits, temperature, alpha)

        # only the student is updated, the teacher stays frozen
        gradients = tape.gradient(loss, student_distilled.trainable_variables)
        optimizer.apply_gradients(zip(gradients, student_distilled.trainable_variables))
        
        epoch_loss += loss.numpy()
        
        # train accuracy comes from the dropout-on forward pass, so it reads lower than an eval pass would
        predictions = tf.argmax(student_logits, axis=1)
        epoch_acc += tf.reduce_mean(tf.cast(tf.equal(predictions, y_batch), tf.float32)).numpy()
    
    avg_loss = epoch_loss / steps_per_epoch
    avg_acc = epoch_acc / steps_per_epoch
    
    # evaluated at t=1, temperature is a training-time device only
    test_logits = student_distilled(x_test, training=False)
    test_preds = tf.argmax(test_logits, axis=1)
    test_acc = tf.reduce_mean(tf.cast(tf.equal(test_preds, y_test), tf.float32)).numpy() * 100
    
    distill_losses.append(avg_loss)
    distill_accs.append(test_acc)
    
    print(f"loss: {avg_loss:.4f}, train acc: {avg_acc*100:.2f}%, test acc: {test_acc:.2f}%")

print(f"\nfinal distilled student accuracy: {distill_accs[-1]:.2f}%")

##  Train Baseline Student (No Distillation)

In [ ]:
# control: same student architecture, optimizer, epochs and batch size, trained on hard labels only
print("training baseline student (no distillation)...")

In [ ]:
student_baseline = create_student_model()
student_baseline.compile(
    optimizer=keras.optimizers.Adam(learning_rate=0.001),
    loss=keras.losses.SparseCategoricalCrossentropy(from_logits=True),
    metrics=['accuracy'])

In [ ]:
history_baseline = student_baseline.fit(
    x_train, y_train,
    validation_data=(x_test, y_test),
    epochs=10,
    batch_size=batch_size,
    verbose=1)

In [ ]:
baseline_losses = history_baseline.history['loss']
baseline_accs = [acc * 100 for acc in history_baseline.history['val_accuracy']]

print(f"\nfinal baseline student accuracy: {baseline_accs[-1]:.2f}%")

## Results Comparison

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

axes[0].plot(teacher_accs, marker='o', label='teacher (large)', linewidth=2)
axes[0].plot(distill_accs, marker='s', label='student (distilled)', linewidth=2)
axes[0].plot(baseline_accs, marker='^', label='student (baseline)', linewidth=2)
axes[0].set_title('test accuracy over training', fontsize=14, fontweight='bold')
axes[0].set_xlabel('epoch')
axes[0].set_ylabel('accuracy (%)')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

In [ ]:
axes[1].plot(teacher_losses, marker='o', label='teacher', linewidth=2)
axes[1].plot(distill_losses, marker='s', label='student (distilled)', linewidth=2)
axes[1].plot(baseline_losses, marker='^', label='student (baseline)', linewidth=2)
axes[1].set_title('training loss', fontsize=14, fontweight='bold')
axes[1].set_xlabel('epoch')
axes[1].set_ylabel('loss')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('distillation_results.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
print("final results:")
print(f"teacher (large model): {teacher_accs[-1]:.2f}%")
print(f"student with distillation: {distill_accs[-1]:.2f}%")
print(f"student without distillation: {baseline_accs[-1]:.2f}%")

In [ ]:
print(f"distillation improvement: +{distill_accs[-1] - baseline_accs[-1]:.2f}%")
print(f"gap to teacher (with distillation): {teacher_accs[-1] - distill_accs[-1]:.2f}%")
print(f"gap to teacher (baseline): {teacher_accs[-1] - baseline_accs[-1]:.2f}%")

In [ ]:
print(f"model size reduction: {(1-student_params/teacher_params)*100:.1f}%")
print(f"performance retained: {(distill_accs[-1]/teacher_accs[-1])*100:.1f}%")

## Inference Speed Comparison

In [ ]:
# wall clock latency per batch, the deployment cost a smaller student is meant to cut
def measure_inference_time(model, x_test, num_batches=50):
    times = []
    
    for i in range(num_batches):
        start_idx = i * batch_size
        end_idx = start_idx + batch_size
        
        if end_idx > len(x_test):
            break
            
        x_batch = x_test[start_idx:end_idx]
        
        start_time = time.time()
        _ = model(x_batch, training=False)
        end_time = time.time()
        
        times.append(end_time - start_time)
    
    return np.mean(times) * 1000

In [ ]:
teacher_time = measure_inference_time(teacher, x_test)
student_time = measure_inference_time(student_distilled, x_test)

In [ ]:
print("\ninference speed comparison:")
print(f"teacher inference time: {teacher_time:.2f} ms/batch")
print(f"student inference time: {student_time:.2f} ms/batch")
print(f"speedup: {teacher_time/student_time:.2f}x faster")
print(f"\nstudent model is {teacher_time/student_time:.1f}x faster with only "
      f"{teacher_accs[-1] - distill_accs[-1]:.2f}% accuracy drop")

## Visualize Predictions

In [ ]:
num_samples = 5
test_images = x_test[:batch_size]
test_labels = y_test[:batch_size]

teacher_preds = tf.argmax(teacher(test_images, training=False), axis=1).numpy()
distilled_preds = tf.argmax(student_distilled(test_images, training=False), axis=1).numpy()
baseline_preds = tf.argmax(student_baseline(test_images, training=False), axis=1).numpy()

In [ ]:
fig, axes = plt.subplots(3, 5, figsize=(15, 9))

for i in range(5):
    img = test_images[i].squeeze()
    true_label = test_labels[i]
    
    for row in range(3):
        axes[row, i].imshow(img, cmap='gray')
        axes[row, i].axis('off')
    
    t_pred = teacher_preds[i]
    color = 'green' if t_pred == true_label else 'red'
    axes[0, i].set_title(f'teacher: {t_pred}\ntrue: {true_label}', color=color, fontweight='bold')
    
    d_pred = distilled_preds[i]
    color = 'green' if d_pred == true_label else 'red'
    axes[1, i].set_title(f'distilled: {d_pred}\ntrue: {true_label}', color=color, fontweight='bold')
    
    b_pred = baseline_preds[i]
    color = 'green' if b_pred == true_label else 'red'
    axes[2, i].set_title(f'baseline: {b_pred}\ntrue: {true_label}', color=color, fontweight='bold')

axes[0, 0].set_ylabel('teacher', fontsize=12, fontweight='bold')
axes[1, 0].set_ylabel('distilled\nstudent', fontsize=12, fontweight='bold')
axes[2, 0].set_ylabel('baseline\nstudent', fontsize=12, fontweight='bold')

plt.tight_layout()
plt.savefig('prediction_comparison.png', dpi=150, bbox_inches='tight')
plt.show()